# **Hate Speech Classification**

Tujuan project: Membangun model klasifikasi biner menggunakan TF-IDF dan Support Vector Machine yang mampu mengklasifikasikan teks berbahasa Indonesia menjadi kelas hate speech dan non-hate speech.


## **Import Library**

In [1]:
!pip install scikit-learn

In [1]:
import pandas as pd
import numpy as np
import csv

from sklearn.model_selection import train_test_split

## **Load Dataset**

In [2]:
# Baca dataset
df = pd.read_csv('../dataset/processed/data_preprocessing.csv', encoding='latin-1')

print("Jumlah data hasil preprocessing:", len(df))

Jumlah data hasil preprocessing: 13023


In [3]:
# Menampilkan informasi struktur dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13023 entries, 0 to 13022
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Tweet          13023 non-null  object
 1   HS             13023 non-null  int64 
 2   cleanpunct     13023 non-null  object
 3   lowercase      13023 non-null  object
 4   tokens         13023 non-null  object
 5   normalization  13023 non-null  object
 6   stopword       13023 non-null  object
 7   stemming       13023 non-null  object
dtypes: int64(1), object(7)
memory usage: 814.1+ KB


In [4]:
# Menampilkan 5 baris pertama
df.head()

,Tweet,HS,cleanpunct,lowercase,tokens,normalization,stopword,stemming
0,- disaat semua cowok berusaha melacak perhatia...,1,disaat semua cowok berusaha melacak perhatian ...,disaat semua cowok berusaha melacak perhatian ...,"['disaat', 'semua', 'cowok', 'berusaha', 'mela...","['di', 'saat', 'semua', 'cowok', 'berusaha', '...","['semua', 'cowok', 'berusaha', 'melacak', 'per...",semua cowok usaha lacak perhati gue kamu lanta...
1,RT USER: USER siapa yang telat ngasih tau elu?...,0,USER siapa yang telat ngasih tau elu edan sara...,user siapa yang telat ngasih tau elu edan sara...,"['user', 'siapa', 'yang', 'telat', 'ngasih', '...","['pengguna', 'siapa', 'yang', 'telat', 'member...","['pengguna', 'siapa', 'telat', 'memberi', 'tau...",guna siapa telat beri tau kamu edan sarap gue ...
2,"41. Kadang aku berfikir, kenapa aku tetap perc...",0,Kadang aku berfikir kenapa aku tetap percaya p...,kadang aku berfikir kenapa aku tetap percaya p...,"['kadang', 'aku', 'berfikir', 'kenapa', 'aku',...","['kadang', 'aku', 'berpikir', 'kenapa', 'aku',...","['kadang', 'aku', 'berpikir', 'aku', 'tetap', ...",kadang aku pikir aku tetap percaya tuhan padah...
3,USER USER AKU ITU AKU\n\nKU TAU MATAMU SIPIT T...,0,USER AKU ITU AKU KU TAU MATAMU SIPIT TAPI DILI...,user aku itu aku ku tau matamu sipit tapi dili...,"['user', 'aku', 'itu', 'aku', 'ku', 'tau', 'ma...","['pengguna', 'aku', 'itu', 'aku', 'aku', 'tau'...","['pengguna', 'aku', 'aku', 'aku', 'tau', 'mata...",guna aku aku aku tau mata sipit lihat mana aku
4,USER USER Kaum cebong kapir udah keliatan dong...,1,USER Kaum cebong kapir udah keliatan dongoknya...,user kaum cebong kapir udah keliatan dongoknya...,"['user', 'kaum', 'cebong', 'kapir', 'udah', 'k...","['pengguna', 'kaum', 'cebong', 'kafir', 'sudah...","['pengguna', 'kaum', 'cebong', 'kafir', 'kelih...",guna kaum cebong kafir lihat dongok awal tamba...


## **Splitting Data**

In [5]:
# Memeriksa teks duplikat dengan label yang berbeda untuk mencegah data leakge
label_per_teks = df.groupby('stemming')['HS'].nunique()
teks_bermasalah = label_per_teks[label_per_teks > 1].index

print("Jumlah teks dengan label yang berbeda:", len(teks_bermasalah))

if len(teks_bermasalah) > 0:
    print("Contoh teks dengan label berbeda:")
    display(df[df['stemming'].isin(teks_bermasalah)].sort_values('stemming').head(50))

Jumlah teks dengan label yang berbeda: 5
Contoh teks dengan label berbeda:


,Tweet,HS,cleanpunct,lowercase,tokens,normalization,stopword,stemming
182,USER USER USER USER USER USER USER USER USER U...,0,USER,user,['user'],['pengguna'],['pengguna'],guna
8003,RT USER: USER USER USER USER USER USER USER US...,0,USER,user,['user'],['pengguna'],['pengguna'],guna
6975,USER Haruss,0,USER Haruss,user haruss,"['user', 'haruss']","['pengguna', 'harus']",['pengguna'],guna
6286,RT USER: USER USER USER USER USER USER USER US...,0,USER,user,['user'],['pengguna'],['pengguna'],guna
3820,USER USER USER USER USER USER USER USER USER U...,0,USER,user,['user'],['pengguna'],['pengguna'],guna
5408,RT USER: USER USER USER USER USER USER USER US...,0,USER,user,['user'],['pengguna'],['pengguna'],guna
488,USER USER USER USER USER USER USER USER USER U...,0,USER,user,['user'],['pengguna'],['pengguna'],guna
376,USER USER USER USER USER USER USER USER USER U...,0,USER,user,['user'],['pengguna'],['pengguna'],guna
317,USER USER USER USER USER USER USER USER USER U...,0,USER,user,['user'],['pengguna'],['pengguna'],guna
288,USER USER USER USER USER USER USER USER USER U...,1,USER,user,['user'],['pengguna'],['pengguna'],guna


In [6]:
# Hitung jumlah baris yang akan dihapus
mask_konflik = df['stemming'].isin(teks_bermasalah)

print("Jumlah kelompok teks bermasalah:", len(teks_bermasalah))
print("Jumlah baris yang akan dihapus:", mask_konflik.sum())

# Hapus seluruh baris dari kelompok bermasalah dikarenakan tweet tersebut tidak dapat dipastikan label yang benar
df = df.loc[~mask_konflik].copy()
print("Jumlah data setelah dihapus:", len(df))

# Memastikan tidak ada konflik label tersisa
konflik_sisa = (df.groupby('stemming')['HS'].nunique().gt(1).sum())
print("Jumlah konflik label tersisa:", konflik_sisa)

Jumlah kelompok teks bermasalah: 5
Jumlah baris yang akan dihapus: 20
Jumlah data setelah dihapus: 13003
Jumlah konflik label tersisa: 0


In [7]:
# Menghapus duplikasi berdasarkan teks stemming
jumlah_duplikat = df.duplicated(subset=['stemming'], keep='first').sum()

df = df.drop_duplicates(subset=['stemming'], keep='first').copy()

print("Jumlah duplikasi yang dihapus:", jumlah_duplikat)
print("Jumlah data final:", len(df))

# Distribusi label HS
print("Distribusi label HS:")
distribusi_hs = pd.DataFrame({
    'Jumlah Data': df['HS'].value_counts().sort_index(),
    'Persentase (%)': (df['HS'].value_counts(normalize=True).sort_index() * 100).round(2)
})

distribusi_hs

Jumlah duplikasi yang dihapus: 151
Jumlah data final: 12852
Distribusi label HS:


,Jumlah Data,Persentase (%)
HS,,
0,7421,57.74
1,5431,42.26


In [9]:
# Fitur dan target
X = df['stemming']
y = df['HS']

# Split training 80% dan testing 20%
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Simpan hasil splitting
train_df = pd.DataFrame({
    'stemming': X_train,
    'HS': y_train
})

test_df = pd.DataFrame({
    'stemming': X_test,
    'HS': y_test
})

# Jumlah data
print("Jumlah data setiap split:")
print(f"Training   : {len(train_df)} ({len(train_df)/len(df)*100:.2f}%)")
print(f"Testing    : {len(test_df)} ({len(test_df)/len(df)*100:.2f}%)")
print(f"Total      : {len(train_df) + len(test_df)}")

# Simpan ke CSV
train_df.rename(columns={'stemming': 'Tweet'}).to_csv('../dataset/processed/data_train.csv', index=False)
test_df.rename(columns={'stemming': 'Tweet'}).to_csv('../dataset/processed/data_test.csv', index=False)

print("\nFile splitting berhasil disimpan")

Jumlah data setiap split:
Training   : 10281 (80.00%)
Testing    : 2571 (20.00%)
Total      : 12852

File splitting berhasil disimpan


In [10]:
# Cek duplikasi lintas split
train_text = set(train_df['stemming'])
test_text = set(test_df['stemming'])

print("Cek duplikasi lintas split:")
print("Train - Test      :", len(train_text & test_text))

Cek duplikasi lintas split:
Train - Test      : 0


In [11]:
# Cek distribusi kelas HS setiap split
hasil_distribusi = []

for nama, subset in [
    ('Training', train_df),
    ('Testing', test_df)
]:
    jumlah = subset['HS'].value_counts().sort_index()
    persentase = (subset['HS'].value_counts(normalize=True).sort_index() * 100)

    for label in [0, 1]:
        hasil_distribusi.append({
            'Subset': nama,
            'Label HS': {
                0: 'Non-Hate Speech',
                1: 'Hate Speech'
            }[label],
            'Jumlah Data': jumlah.get(label, 0),
            'Persentase (%)': round(persentase.get(label, 0), 2)
        })

distribusi_hs = pd.DataFrame(hasil_distribusi)
distribusi_hs

,Subset,Label HS,Jumlah Data,Persentase (%)
0,Training,Non-Hate Speech,5936,57.74
1,Training,Hate Speech,4345,42.26
2,Testing,Non-Hate Speech,1485,57.76
3,Testing,Hate Speech,1086,42.24
